# 05 - Construcción del dataset NER para detección de platos

## Objetivo

Este notebook construye el primer dataset débil para entrenar un modelo NER de detección de platos.

Parte de los candidatos refinados generados en el Notebook 04:

- `weak_dish_candidates_v2.jsonl`
- `dish_lexicon_seed_v1.csv`
- corpus Yelp original

El objetivo es transformar candidatos de plato en spans dentro del texto:

```json
{
  "text": "The crab legs were amazing",
  "entities": [
    {
      "start": 4,
      "end": 13,
      "label": "DISH",
      "text": "crab legs"
    }
  ]
}

```
Este dataset todavía no es un gold standard manual. Es un dataset débil (weakly supervised) que servirá como base para el futuro entrenamiento NER.


In [3]:
# ============================================================
# 01. Imports y configuración base
# ============================================================

from pathlib import Path
import json
import random
import re
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 300)

print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


In [4]:
# ============================================================
# 02. Detectar entorno de ejecución
# ============================================================

import os

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IS_KAGGLE:
    ENV_NAME = "kaggle"
elif IS_COLAB:
    ENV_NAME = "colab"
else:
    ENV_NAME = "local"

print("Entorno detectado:", ENV_NAME)

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "hidden_gems_ai"
    DATA_DIR = PROJECT_DIR / "data"

elif IS_COLAB:
    from google.colab import drive

    try:
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")
        DATA_DIR = PROJECT_DIR / "data"

    except Exception as e:
        print("No se ha podido montar Drive. Se usará almacenamiento temporal.")
        print("Error:", e)

        PROJECT_DIR = Path("/content/hidden_gems_ai")
        DATA_DIR = PROJECT_DIR / "data"

else:
    PROJECT_DIR = Path.cwd()
    DATA_DIR = PROJECT_DIR / "data"

OUTPUT_DIR = PROJECT_DIR / "outputs" / "dish_ner_dataset_builder"
SPANS_DIR = OUTPUT_DIR / "spans"
BIO_DIR = OUTPUT_DIR / "bio"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
SAMPLES_DIR = OUTPUT_DIR / "samples"

for path in [PROJECT_DIR, DATA_DIR, OUTPUT_DIR, SPANS_DIR, BIO_DIR, ARTIFACTS_DIR, SAMPLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Entorno detectado: kaggle
PROJECT_DIR: /kaggle/working/hidden_gems_ai
DATA_DIR: /kaggle/working/hidden_gems_ai/data
OUTPUT_DIR: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder


In [5]:
# ============================================================
# 03. Diagnóstico de archivos disponibles
# ============================================================

if IS_KAGGLE:
    print("Archivos disponibles en /kaggle/input:")

    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("No estás en Kaggle. Se buscarán archivos en DATA_DIR y PROJECT_DIR.")

Archivos disponibles en /kaggle/input:
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_manual_review_sample_v2_150.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/transformer_sentiment_metrics.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidates_v2.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_exploration_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_translation_gold_seed_300_es_ready_v1.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidate_frequencies_v2.csv
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_lexicon_seed_v1.csv
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/classic_model_metrics.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_food_reviews_corpus_sample_100k_lines.jsonl


In [6]:
# ============================================================
# 04. Localizar archivos necesarios
# ============================================================

REQUIRED_FILES = {
    "corpus": "yelp_food_reviews_corpus_sample_100k_lines.jsonl",
    "weak_candidates_v2": "weak_dish_candidates_v2.jsonl",
    "dish_lexicon": "dish_lexicon_seed_v1.csv",
}

def find_file(filename: str) -> Path:
    candidate_paths = []

    if IS_KAGGLE:
        candidate_paths.extend(list(Path("/kaggle/input").rglob(filename)))

    candidate_paths.extend(list(DATA_DIR.rglob(filename)))
    candidate_paths.extend(list(PROJECT_DIR.rglob(filename)))
    candidate_paths.extend(list(Path.cwd().rglob(filename)))

    candidate_paths = list(dict.fromkeys(candidate_paths))

    if not candidate_paths:
        raise FileNotFoundError(
            f"No se ha encontrado el archivo requerido: {filename}\n"
            "En Kaggle, asegúrate de haberlo añadido desde Add Data."
        )

    return candidate_paths[0]


CORPUS_PATH = find_file(REQUIRED_FILES["corpus"])
WEAK_CANDIDATES_V2_PATH = find_file(REQUIRED_FILES["weak_candidates_v2"])
DISH_LEXICON_PATH = find_file(REQUIRED_FILES["dish_lexicon"])

print("Archivos localizados:")
print("CORPUS_PATH:", CORPUS_PATH)
print("WEAK_CANDIDATES_V2_PATH:", WEAK_CANDIDATES_V2_PATH)
print("DISH_LEXICON_PATH:", DISH_LEXICON_PATH)

Archivos localizados:
CORPUS_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_food_reviews_corpus_sample_100k_lines.jsonl
WEAK_CANDIDATES_V2_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidates_v2.jsonl
DISH_LEXICON_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_lexicon_seed_v1.csv


In [7]:
# ============================================================
# 05. Funciones de carga
# ============================================================

def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    invalid_json_count = 0

    with path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Leyendo {path.name}"):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                invalid_json_count += 1

    print(f"Archivo: {path.name}")
    print(f"Registros cargados: {len(records):,}")
    print(f"Líneas JSON inválidas: {invalid_json_count:,}")

    return pd.DataFrame(records)


def make_json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, set):
        return sorted(list(value))

    return value


def save_jsonl(df_to_save: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in df_to_save.to_dict(orient="records"):
            clean_record = {
                str(k): make_json_safe(v)
                for k, v in record.items()
            }
            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

    print(f"Guardado: {path} ({len(df_to_save):,} registros)")

In [8]:
# ============================================================
# 06. Cargar corpus, candidatos v2 y lexicón
# ============================================================

df_raw = load_jsonl(CORPUS_PATH)
candidates_v2_df = load_jsonl(WEAK_CANDIDATES_V2_PATH)
lexicon_df = pd.read_csv(DISH_LEXICON_PATH)

print("\nCorpus shape:", df_raw.shape)
print("Candidates v2 shape:", candidates_v2_df.shape)
print("Lexicon shape:", lexicon_df.shape)

print("\nCorpus columns:")
print(df_raw.columns.tolist())

print("\nCandidates v2 columns:")
print(candidates_v2_df.columns.tolist())

print("\nLexicon columns:")
print(lexicon_df.columns.tolist())

display(df_raw.head(2))
display(candidates_v2_df.head(5))
display(lexicon_df.head(5))

Leyendo yelp_food_reviews_corpus_sample_100k_lines.jsonl: 0it [00:00, ?it/s]

Archivo: yelp_food_reviews_corpus_sample_100k_lines.jsonl
Registros cargados: 79,270
Líneas JSON inválidas: 0


Leyendo weak_dish_candidates_v2.jsonl: 0it [00:00, ?it/s]

Archivo: weak_dish_candidates_v2.jsonl
Registros cargados: 100,965
Líneas JSON inválidas: 0

Corpus shape: (79270, 20)
Candidates v2 shape: (100965, 16)
Lexicon shape: (5123, 12)

Corpus columns:
['corpus_document_id', 'source_system_code', 'source_dataset', 'source_entity_type', 'source_review_id', 'source_business_id', 'source_user_id', 'text', 'text_normalized', 'language', 'rating_value', 'sentiment_label_from_rating', 'review_date', 'corpus_split', 'task_scope', 'is_training_eligible', 'quality_flags', 'business_metadata', 'source_metrics', 'created_at']

Candidates v2 columns:
['corpus_document_id', 'review_id', 'business_id', 'business_name', 'city', 'state', 'split', 'rating_value', 'sentiment_label', 'candidate_text', 'candidate_sources', 'weak_confidence', 'text_clean', 'candidate_granularity', 'usable_for_ner_weak_label', 'weak_confidence_v2']

Lexicon columns:
['candidate_text', 'candidate_granularity', 'mention_count', 'review_count', 'avg_weak_confidence_v2', 'canonical_d

,corpus_document_id,source_system_code,source_dataset,source_entity_type,source_review_id,source_business_id,source_user_id,text,text_normalized,language,rating_value,sentiment_label_from_rating,review_date,corpus_split,task_scope,is_training_eligible,quality_flags,business_metadata,source_metrics,created_at
0,yelp_2fbfd094613536a7b8c9231b,yelp_open_dataset,yelp_open_dataset,yelp_review,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,mh_-eMZ6K5RLWhZyISBhwA,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...","If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...",en,3.0,neutral,2018-07-07 22:09:11,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 511, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Turning Point of North Wales', 'city': 'North Wales', 'state': 'PA', 'stars_business': 3.0, 'review_count_business': 169, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch', 'Food', 'Juice Bars & Smoothies', 'American (New)', 'Coffee & Tea', 'Sandwiches'], '...","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.623430+00:00
1,yelp_94c5a64cecd4448d105e5c8a,yelp_open_dataset,yelp_open_dataset,yelp_review,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,8g_iMtfSiwikVnbP2etR0A,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",en,3.0,neutral,2014-02-05 20:30:30,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 339, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Kettle Restaurant', 'city': 'Tucson', 'state': 'AZ', 'stars_business': 3.5, 'review_count_business': 47, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch'], 'food_category_tags': ['Breakfast & Brunch', 'Restaurants'], 'food_confidence': 0.95}","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.624430+00:00


,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,candidate_text,candidate_sources,weak_confidence,text_clean,candidate_granularity,usable_for_ner_weak_label,weak_confidence_v2
0,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,3.0,neutral,tamale,[dictionary],0.7,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",dish_family,True,0.75
1,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,lamb curry,"[dictionary, pattern]",0.9,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",dish_variant,True,0.99
2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,korma,[dictionary],0.7,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",dish_family,True,0.75
3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,naan,[dictionary],0.7,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",dish_family,True,0.75
4,yelp_95095495afc3a77ce68723a2,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,train,4.0,positive,salad,[dictionary],0.7,"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m...",dish_family_ambiguous,True,0.65


,candidate_text,candidate_granularity,mention_count,review_count,avg_weak_confidence_v2,canonical_dish_name,dish_family,language,lexicon_source,manual_review_status,include_in_ner_dictionary,manual_review_priority
0,salad,dish_family_ambiguous,5774,5774,0.655040,salad,salad,en,weak_detection_v2,pending,True,2
1,pizza,dish_family,4848,4848,0.756040,pizza,pizza,en,weak_detection_v2,pending,True,2
2,coffee,dish_family_ambiguous,4399,4399,0.655388,coffee,coffee,en,weak_detection_v2,pending,True,2
3,sandwich,dish_family_ambiguous,4359,4359,0.653441,sandwich,sandwich,en,weak_detection_v2,pending,True,2
4,fries,dish_family,4238,4238,0.750566,fries,fries,en,weak_detection_v2,pending,True,2


In [9]:
# ============================================================
# 07. Preparar corpus base
# ============================================================

required_corpus_cols = [
    "corpus_document_id",
    "source_review_id",
    "source_business_id",
    "text",
    "text_normalized",
    "language",
    "rating_value",
    "sentiment_label_from_rating",
    "corpus_split",
]

missing_corpus_cols = [col for col in required_corpus_cols if col not in df_raw.columns]

if missing_corpus_cols:
    raise ValueError(f"Faltan columnas obligatorias en el corpus: {missing_corpus_cols}")

df = df_raw.copy()

df["corpus_document_id"] = df["corpus_document_id"].astype(str)
df["review_id"] = df["source_review_id"].astype(str)
df["business_id"] = df["source_business_id"].astype(str)

df["text_original"] = (
    df["text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["text_clean"] = (
    df["text_normalized"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["language"] = df["language"].astype(str).str.lower().str.strip()

df["rating_value"] = pd.to_numeric(df["rating_value"], errors="coerce")

df["sentiment_label"] = (
    df["sentiment_label_from_rating"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["split"] = (
    df["corpus_split"]
    .astype(str)
    .str.lower()
    .str.strip()
)

valid_splits = {"train", "validation", "test"}
valid_sentiments = {"positive", "neutral", "negative"}

df = df[
    df["split"].isin(valid_splits)
    & df["sentiment_label"].isin(valid_sentiments)
    & df["text_clean"].notna()
    & (df["text_clean"].str.len() > 0)
].copy()

df["text_length_chars"] = df["text_clean"].str.len()

print("Corpus preparado:", df.shape)

display(pd.crosstab(df["split"], df["sentiment_label"]))
display(df.head(3))

Corpus preparado: (79270, 27)


sentiment_label,negative,neutral,positive
split,,,
test,1428,936,5517
train,11701,7894,43945
validation,1449,1005,5395


,corpus_document_id,source_system_code,source_dataset,source_entity_type,source_review_id,source_business_id,source_user_id,text,text_normalized,language,rating_value,sentiment_label_from_rating,review_date,corpus_split,task_scope,is_training_eligible,quality_flags,business_metadata,source_metrics,created_at,review_id,business_id,text_original,text_clean,sentiment_label,split,text_length_chars
0,yelp_2fbfd094613536a7b8c9231b,yelp_open_dataset,yelp_open_dataset,yelp_review,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,mh_-eMZ6K5RLWhZyISBhwA,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...","If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...",en,3.0,neutral,2018-07-07 22:09:11,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 511, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Turning Point of North Wales', 'city': 'North Wales', 'state': 'PA', 'stars_business': 3.0, 'review_count_business': 169, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch', 'Food', 'Juice Bars & Smoothies', 'American (New)', 'Coffee & Tea', 'Sandwiches'], '...","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.623430+00:00,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...","If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...",neutral,train,511
1,yelp_94c5a64cecd4448d105e5c8a,yelp_open_dataset,yelp_open_dataset,yelp_review,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,8g_iMtfSiwikVnbP2etR0A,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",en,3.0,neutral,2014-02-05 20:30:30,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 339, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Kettle Restaurant', 'city': 'Tucson', 'state': 'AZ', 'stars_business': 3.5, 'review_count_business': 47, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch'], 'food_category_tags': ['Breakfast & Brunch', 'Restaurants'], 'food_confidence': 0.95}","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.624430+00:00,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,"Family diner. Had the buffet. Eclectic assortment: a large chick

In [10]:
# ============================================================
# 08. Preparar candidatos v2
# ============================================================

required_candidate_cols = [
    "review_id",
    "candidate_text",
    "candidate_sources",
    "candidate_granularity",
    "weak_confidence_v2",
]

missing_candidate_cols = [col for col in required_candidate_cols if col not in candidates_v2_df.columns]

if missing_candidate_cols:
    raise ValueError(f"Faltan columnas obligatorias en candidatos v2: {missing_candidate_cols}")

def normalize_candidate_text(candidate: str) -> str:
    if not isinstance(candidate, str):
        return ""

    candidate = candidate.lower().strip()
    candidate = re.sub(r"\s+", " ", candidate)
    candidate = candidate.strip(" -_:;,.!?()[]{}\"'")

    return candidate


def safe_sources(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        return [value]

    return []


candidates = candidates_v2_df.copy()

candidates["review_id"] = candidates["review_id"].astype(str)
candidates["candidate_text"] = candidates["candidate_text"].apply(normalize_candidate_text)
candidates["candidate_sources"] = candidates["candidate_sources"].apply(safe_sources)
candidates["candidate_granularity"] = candidates["candidate_granularity"].astype(str)
candidates["weak_confidence_v2"] = pd.to_numeric(candidates["weak_confidence_v2"], errors="coerce").fillna(0.0)

# Eliminar candidatos vacíos
candidates = candidates[candidates["candidate_text"].str.len() > 0].copy()

# Solo candidatos de reviews existentes
valid_review_ids = set(df["review_id"])
before_count = len(candidates)

candidates = candidates[candidates["review_id"].isin(valid_review_ids)].copy()

after_count = len(candidates)

print("Candidatos antes:", before_count)
print("Candidatos después de cruzar con corpus:", after_count)

print("\nGranularidad:")
display(candidates["candidate_granularity"].value_counts())

print("\nConfianza:")
display(candidates["weak_confidence_v2"].describe())

display(candidates.head(10))

Candidatos antes: 100965
Candidatos después de cruzar con corpus: 100965

Granularidad:


candidate_granularity
dish_family              57367
dish_family_ambiguous    29936
dish_variant             13662
Name: count, dtype: int64


Confianza:


count    100965.000000
mean          0.734453
std           0.080255
min           0.600000
25%           0.650000
50%           0.750000
75%           0.750000
max           0.990000
Name: weak_confidence_v2, dtype: float64

,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,candidate_text,candidate_sources,weak_confidence,text_clean,candidate_granularity,usable_for_ner_weak_label,weak_confidence_v2
0,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,3.0,neutral,tamale,[dictionary],0.7,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",dish_family,True,0.75
1,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,lamb curry,"[dictionary, pattern]",0.9,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",dish_variant,True,0.99
2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,korma,[dictionary],0.7,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",dish_family,True,0.75
3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,naan,[dictionary],0.7,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",dish_family,True,0.75
4,yelp_95095495afc3a77ce68723a2,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,train,4.0,positive,salad,[dictionary],0.7,"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m...",dish_family_ambiguous,True,0.65
5,yelp_95095495afc3a77ce68723a2,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,train,4.0,positive,sandwich,[dictionary],0.7,"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m...",dish_family_ambiguous,True,0.65
6,yelp_95095495afc3a77ce68723a2,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,train,4.0,positive,sandwiches,[dictionary],0.7,"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m...",dish_family,True,0.75
7,yelp_6ce3d2a264cfcdddc59ac240,_ZeMknuYdlQcUqng_Im3yg,LHSTtnW3YHCeUkRDGyJOyw,Fries Rebellion,Quakertown,PA,train,5.0,positive,wings,[dictionary],0.7,"Amazingly amazing wings and homemade bleu cheese. Had the ribeye: tender, perfectly prepared, delicious. Nice selection of craft beers. Would DEFINITELY recommend checking out this hidden gem.",dish_family,True,0.75
8,yelp_1e7a38ac7da3467ee929e559,pUycOfUwM8vqX7KjRRhUEA,gebiRewfieSdtt17PTW6Zg,Hibachi Steak House & Sushi Bar,Santa Barbara,CA,validation,3.0,neutral,sushi,[dictionary],0.7,Had a party of 6 here for hibachi. Our waitress brought our separate sushi orders on one plate so we couldn't really 

## Estrategia de construcción de spans

Para construir el dataset NER se buscarán los candidatos dentro del texto de cada review.

Se crearán dos versiones:

1. **all_v2**  
   Incluye todos los candidatos refinados v2.

2. **high_precision**  
   Incluye solo candidatos más seguros para entrenamiento inicial:
   - `dish_family`
   - `dish_variant`
   - confianza mínima suficiente
   - excluye inicialmente `dish_family_ambiguous`

La versión `high_precision` será la recomendada para el primer entrenamiento NER, porque prioriza precisión sobre cobertura.

In [11]:
# ============================================================
# 09. Crear subconjuntos all_v2 y high_precision
# ============================================================

HIGH_PRECISION_GRANULARITIES = {
    "dish_family",
    "dish_variant",
}

HIGH_PRECISION_MIN_CONFIDENCE = 0.70

candidates_all_v2 = candidates.copy()

candidates_high_precision = candidates[
    candidates["candidate_granularity"].isin(HIGH_PRECISION_GRANULARITIES)
    & (candidates["weak_confidence_v2"] >= HIGH_PRECISION_MIN_CONFIDENCE)
].copy()

print("Candidatos all_v2:", len(candidates_all_v2))
print("Candidatos high_precision:", len(candidates_high_precision))

print("\nGranularidad high_precision:")
display(candidates_high_precision["candidate_granularity"].value_counts())

print("\nTop candidatos high_precision:")
display(
    candidates_high_precision
    .groupby(["candidate_text", "candidate_granularity"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

Candidatos all_v2: 100965
Candidatos high_precision: 70880

Granularidad high_precision:


candidate_granularity
dish_family     57218
dish_variant    13662
Name: count, dtype: int64


Top candidatos high_precision:


,candidate_text,candidate_granularity,count
3317,pizza,dish_family,4848
2066,fries,dish_family,4238
3998,shrimp,dish_family,4085
665,burger,dish_family,3966
4411,steak,dish_family,2648
4526,sushi,dish_family,2567
4598,tacos,dish_family,2461
725,burgers,dish_family,1995
3884,sandwiches,dish_family,1945
2454,ice cream,dish_family,1775


In [12]:
# ============================================================
# 10. Funciones de búsqueda de spans
# ============================================================

def build_candidate_regex(candidate_text: str):
    """
    Construye un patrón robusto para encontrar candidatos como palabras completas.
    """

    candidate_text = normalize_candidate_text(candidate_text)

    if not candidate_text:
        return None

    escaped = re.escape(candidate_text)

    # Límites aproximados de palabra.
    # Evita detectar pizza dentro de pizzaria, etc.
    pattern = r"(?<![a-zA-Z])" + escaped + r"(?![a-zA-Z])"

    return re.compile(pattern, flags=re.IGNORECASE)


def find_candidate_spans_in_text(text: str, candidate_text: str) -> list[dict]:
    """
    Encuentra todas las apariciones exactas de candidate_text en text.
    Devuelve offsets start/end.
    """

    if not isinstance(text, str):
        return []

    candidate_text = normalize_candidate_text(candidate_text)

    if not candidate_text:
        return []

    regex = build_candidate_regex(candidate_text)

    if regex is None:
        return []

    spans = []

    for match in regex.finditer(text):
        start = int(match.start())
        end = int(match.end())

        matched_text = text[start:end]

        spans.append({
            "start": start,
            "end": end,
            "text": matched_text,
        })

    return spans


def spans_overlap(span_a: dict, span_b: dict) -> bool:
    return not (span_a["end"] <= span_b["start"] or span_b["end"] <= span_a["start"])


GRANULARITY_RANK = {
    "dish_variant": 3,
    "dish_family": 2,
    "dish_family_ambiguous": 1,
}


def resolve_overlapping_spans(spans: list[dict]) -> list[dict]:
    """
    Resuelve solapamientos priorizando:
    1. spans más largos;
    2. mayor confianza;
    3. mayor granularidad;
    4. posición más temprana.
    """

    if not spans:
        return []

    sorted_spans = sorted(
        spans,
        key=lambda s: (
            -(s["end"] - s["start"]),
            -float(s.get("weak_confidence_v2", 0.0)),
            -GRANULARITY_RANK.get(s.get("candidate_granularity", ""), 0),
            s["start"],
        )
    )

    selected = []

    for span in sorted_spans:
        if not any(spans_overlap(span, existing) for existing in selected):
            selected.append(span)

    selected = sorted(selected, key=lambda s: (s["start"], s["end"]))

    return selected

In [13]:
# ============================================================
# 11. Construir documentos con spans
# ============================================================

review_text_map = df.set_index("review_id")["text_clean"].to_dict()
review_doc_id_map = df.set_index("review_id")["corpus_document_id"].to_dict()
review_split_map = df.set_index("review_id")["split"].to_dict()
review_sentiment_map = df.set_index("review_id")["sentiment_label"].to_dict()
review_rating_map = df.set_index("review_id")["rating_value"].to_dict()
review_business_map = df.set_index("review_id")["business_id"].to_dict()

def build_span_documents(candidates_source_df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """
    Construye documentos con spans DISH a partir de candidatos.
    """

    docs = []

    grouped = candidates_source_df.groupby("review_id")

    for review_id, group in tqdm(grouped, desc=f"Construyendo spans {dataset_name}"):
        text = review_text_map.get(review_id)

        if not isinstance(text, str) or not text.strip():
            continue

        raw_spans = []

        # Evitar procesar duplicados exactos de candidato dentro de la misma review
        candidate_items = (
            group[
                [
                    "candidate_text",
                    "candidate_sources",
                    "candidate_granularity",
                    "weak_confidence_v2"
                ]
            ]
            .drop_duplicates(subset=["candidate_text"])
            .to_dict(orient="records")
        )

        for item in candidate_items:
            candidate_text = normalize_candidate_text(item["candidate_text"])

            found_spans = find_candidate_spans_in_text(text, candidate_text)

            for found in found_spans:
                raw_spans.append({
                    "start": int(found["start"]),
                    "end": int(found["end"]),
                    "label": "DISH",
                    "text": found["text"],
                    "normalized_text": candidate_text,
                    "candidate_granularity": item["candidate_granularity"],
                    "candidate_sources": item["candidate_sources"],
                    "weak_confidence_v2": float(item["weak_confidence_v2"]),
                })

        resolved_spans = resolve_overlapping_spans(raw_spans)

        if not resolved_spans:
            continue

        docs.append({
            "corpus_document_id": review_doc_id_map.get(review_id),
            "review_id": review_id,
            "business_id": review_business_map.get(review_id),
            "split": review_split_map.get(review_id),
            "rating_value": make_json_safe(review_rating_map.get(review_id)),
            "sentiment_label": review_sentiment_map.get(review_id),
            "text": text,
            "entities": resolved_spans,
            "entity_count": len(resolved_spans),
            "dataset_name": dataset_name,
            "labeling_method": "weak_exact_match_v2",
            "human_review_status": "not_reviewed",
        })

    return pd.DataFrame(docs)


span_docs_all_v2_df = build_span_documents(candidates_all_v2, "dish_ner_weak_all_v2")
span_docs_hp_df = build_span_documents(candidates_high_precision, "dish_ner_weak_high_precision")

print("Documentos con spans all_v2:", len(span_docs_all_v2_df))
print("Documentos con spans high_precision:", len(span_docs_hp_df))

display(span_docs_hp_df.head(5))

Construyendo spans dish_ner_weak_all_v2:   0%|          | 0/49189 [00:00<?, ?it/s]

Construyendo spans dish_ner_weak_high_precision:   0%|          | 0/40845 [00:00<?, ?it/s]

Documentos con spans all_v2: 49189
Documentos con spans high_precision: 40845


,corpus_document_id,review_id,business_id,split,rating_value,sentiment_label,text,entities,entity_count,dataset_name,labeling_method,human_review_status
0,yelp_7b29d6cf1caf882c03e5b353,--46suwqJVAnei8ng2TeDw,qlC5ojQRLe26shP1Kt7hQg,train,3.0,neutral,"I'm giving this Waffle house 3 stars because on some days it's a four star, but sometimes it's a two star at best. The food quality is always good, customer service is what differs on each visit. Somedays you walk in and are greeted with ""Welcome to Waffle House!"" while other times your ignored ...","[{'start': 16, 'end': 22, 'label': 'DISH', 'text': 'Waffle', 'normalized_text': 'waffle', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 250, 'end': 256, 'label': 'DISH', 'text': 'Waffle', 'normalized_text': 'waffle', 'candidat...",2,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
1,yelp_d62c2a178e2e515a1ff5753f,--Gdv6LOoo7UtL6_A5iiiQ,S2Ho8yLxhKAa26pBAm6rxA,validation,4.0,positive,"This place was fantastic. We were wandering around the French Quarter and stopped in for a drink, which turned into appetizers. We sat at the bar, by the oyster shucking station. The staff was fun and playful. We all had moscow mule, fantastic. We ordered the chargrilled bacon, cheese oysters, t...","[{'start': 154, 'end': 160, 'label': 'DISH', 'text': 'oyster', 'normalized_text': 'oyster', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 260, 'end': 277, 'label': 'DISH', 'text': 'chargrilled bacon', 'normalized_text': 'charg...",4,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
2,yelp_f27e049e9e102a3631b89743,--IuxB6TbaUz8ogNHe9DNQ,34Eqv8jXgxg_EEwcsNgeeg,test,1.0,negative,I usually come to this place a lot and enjoy there smoothies and acai bowls all the time. Sometimes the wait can get pretty bad but nothing too crazy until today. I ordered a Hercules smoothie which I get all the time and the wait is usually 5-15 mins. Today I waited 30 mins for a smoothie. I go...,"[{'start': 175, 'end': 192, 'label': 'DISH', 'text': 'Hercules smoothie', 'normalized_text': 'hercules smoothie', 'candidate_granularity': 'dish_variant', 'candidate_sources': ['pattern'], 'weak_confidence_v2': 0.7}, {'start': 282, 'end': 290, 'label': 'DISH', 'text': 'smoothie', 'normalized_tex...",5,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
3,yelp_c9c63e2c13324a382bbf1118,--N4YcoxfGNd4O8-zbFYjA,sdWuLh-auc0nC2Jy6_26AQ,train,5.0,positive,"A comfortable atmosphere, excellent menu, food well prepared and reasonable, good service. My favorite is Taco Tuesday. Their tacos are the very best.","[{'start': 106, 'end': 118, 'label': 'DISH', 'text': 'Taco Tuesday', 'normalized_text': 'taco tuesday', 'candidate_granularity': 'dish_variant', 'candidate_sources': ['pattern'], 'weak_confidence_v2': 0.7}, {'start': 126, 'end': 131, 'label': 'DISH', 'text': 'tacos', 'normalized_text': 'tacos', ...",2,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
4,yelp_9eca04c4f2c89325942c022f,--NOhHyIi3EeDQpdJsDLVA,a1u9Bxrq_fZxl2pgqQUcJA,train,5.0,positive,"FAVORITE RESTAURANT IN NOLA! (so far anyway lol) Great service, amazing food. we went twice in in two days. The chicken and waffle is absolutely delicious. Their fig and blue cheese dip with pita chips are awesome. Honestly, everything we tried was really good --- including their drinks. A great...","[{'start': 124, 'end': 130, 'label': 'DISH', 'text': 'waffle', 'normalized_text': 'waffle', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}]",1,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed


In [14]:
# ============================================================
# 12. QA de spans
# ============================================================

def validate_spans(span_docs_df: pd.DataFrame, dataset_name: str) -> dict:
    total_docs = len(span_docs_df)
    total_entities = 0
    invalid_offsets = 0
    empty_entities = 0
    mismatch_text = 0
    overlap_docs = 0

    entity_text_counter = Counter()
    granularity_counter = Counter()

    for row in span_docs_df.itertuples(index=False):
        text = row.text
        entities = row.entities

        if not isinstance(entities, list):
            continue

        total_entities += len(entities)

        sorted_entities = sorted(entities, key=lambda e: (e["start"], e["end"]))

        # Overlap check
        for i in range(1, len(sorted_entities)):
            if spans_overlap(sorted_entities[i - 1], sorted_entities[i]):
                overlap_docs += 1
                break

        for ent in entities:
            start = ent.get("start")
            end = ent.get("end")

            if start is None or end is None or start < 0 or end > len(text) or start >= end:
                invalid_offsets += 1
                continue

            extracted = text[start:end]

            if not extracted.strip():
                empty_entities += 1

            if extracted != ent.get("text"):
                mismatch_text += 1

            entity_text_counter[ent.get("normalized_text", "").lower()] += 1
            granularity_counter[ent.get("candidate_granularity", "unknown")] += 1

    summary = {
        "dataset_name": dataset_name,
        "total_docs": int(total_docs),
        "total_entities": int(total_entities),
        "avg_entities_per_doc": float(total_entities / total_docs) if total_docs else 0.0,
        "invalid_offsets": int(invalid_offsets),
        "empty_entities": int(empty_entities),
        "mismatch_text": int(mismatch_text),
        "overlap_docs": int(overlap_docs),
        "top_entities": [
            {"entity": k, "count": int(v)}
            for k, v in entity_text_counter.most_common(30)
        ],
        "granularity_counts": {
            str(k): int(v)
            for k, v in granularity_counter.items()
        },
    }

    return summary


qa_all_v2 = validate_spans(span_docs_all_v2_df, "dish_ner_weak_all_v2")
qa_hp = validate_spans(span_docs_hp_df, "dish_ner_weak_high_precision")

print("QA all_v2:")
print(json.dumps(qa_all_v2, indent=2, ensure_ascii=False)[:3000])

print("\nQA high_precision:")
print(json.dumps(qa_hp, indent=2, ensure_ascii=False)[:3000])

QA all_v2:
{
  "dataset_name": "dish_ner_weak_all_v2",
  "total_docs": 49189,
  "total_entities": 138433,
  "avg_entities_per_doc": 2.8143080770090876,
  "invalid_offsets": 0,
  "empty_entities": 0,
  "mismatch_text": 0,
  "overlap_docs": 0,
  "top_entities": [
    {
      "entity": "pizza",
      "count": 9664
    },
    {
      "entity": "salad",
      "count": 7459
    },
    {
      "entity": "burger",
      "count": 6790
    },
    {
      "entity": "coffee",
      "count": 6638
    },
    {
      "entity": "fries",
      "count": 5823
    },
    {
      "entity": "sandwich",
      "count": 5606
    },
    {
      "entity": "shrimp",
      "count": 5525
    },
    {
      "entity": "sushi",
      "count": 4878
    },
    {
      "entity": "soup",
      "count": 4061
    },
    {
      "entity": "steak",
      "count": 3519
    },
    {
      "entity": "tacos",
      "count": 3422
    },
    {
      "entity": "bacon",
      "count": 3081
    },
    {
      "entity": "ice cream",
  

In [15]:
# ============================================================
# 13. Distribución de datasets de spans
# ============================================================

def show_span_dataset_distribution(span_docs_df: pd.DataFrame, name: str):
    print("=" * 100)
    print(name)
    print("=" * 100)

    print("Documentos:", len(span_docs_df))
    print("Entidades:", int(span_docs_df["entity_count"].sum()) if len(span_docs_df) else 0)

    print("\nSplit counts:")
    display(span_docs_df["split"].value_counts())

    print("\nSentiment counts:")
    display(span_docs_df["sentiment_label"].value_counts())

    print("\nSplit x sentiment:")
    display(pd.crosstab(span_docs_df["split"], span_docs_df["sentiment_label"]))

    print("\nEntity count per doc:")
    display(span_docs_df["entity_count"].describe())


show_span_dataset_distribution(span_docs_all_v2_df, "ALL V2")
show_span_dataset_distribution(span_docs_hp_df, "HIGH PRECISION")

ALL V2
Documentos: 49189
Entidades: 138433

Split counts:


split
train         39427
test           4897
validation     4865
Name: count, dtype: int64


Sentiment counts:


sentiment_label
positive    34497
negative     8229
neutral      6463
Name: count, dtype: int64


Split x sentiment:


...

sentiment_label
positive    28607
negative     6769
neutral      5469
Name: count, dtype: int64


Split x sentiment:


sentiment_label,negative,neutral,positive
split,,,
test,639,525,2906
train,5444,4362,22876
validation,686,582,2825



Entity count per doc:


count    40845.000000
mean         2.403183
std          2.023043
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         37.000000
Name: entity_count, dtype: float64

In [16]:
# ============================================================
# 14. Añadir documentos negativos controlados
# ============================================================

def add_controlled_negative_documents(
    positive_span_docs_df: pd.DataFrame,
    corpus_df: pd.DataFrame,
    dataset_name: str,
    negative_ratio: float = 0.10,
    max_negatives_per_split: int = 1500,
    random_state: int = RANDOM_STATE
) -> pd.DataFrame:
    """
    Añade una pequeña cantidad de documentos sin entidades.
    Esto ayuda al NER a aprender ejemplos con todo O, pero se mantiene controlado
    para no introducir demasiados falsos negativos.
    """

    positive_review_ids = set(positive_span_docs_df["review_id"])

    negative_pool = corpus_df[~corpus_df["review_id"].isin(positive_review_ids)].copy()

    negative_docs = []

    for split_name in ["train", "validation", "test"]:
        positive_split_count = (positive_span_docs_df["split"] == split_name).sum()

        target_n = int(positive_split_count * negative_ratio)
        target_n = min(target_n, max_negatives_per_split)

        split_pool = negative_pool[negative_pool["split"] == split_name].copy()

        if len(split_pool) == 0 or target_n <= 0:
            continue

        sample_n = min(target_n, len(split_pool))

        sampled = split_pool.sample(
            n=sample_n,
            random_state=random_state
        )

        for row in sampled.itertuples(index=False):
            negative_docs.append({
                "corpus_document_id": row.corpus_document_id,
                "review_id": row.review_id,
                "business_id": row.business_id,
                "split": row.split,
                "rating_value": make_json_safe(row.rating_value),
                "sentiment_label": row.sentiment_label,
                "text": row.text_clean,
                "entities": [],
                "entity_count": 0,
                "dataset_name": dataset_name,
                "labeling_method": "controlled_negative_no_weak_dish_span",
                "human_review_status": "not_reviewed",
            })

    negative_docs_df = pd.DataFrame(negative_docs)

    combined = pd.concat(
        [positive_span_docs_df, negative_docs_df],
        ignore_index=True
    )

    combined = combined.sample(
        frac=1.0,
        random_state=random_state
    ).reset_index(drop=True)

    return combined


span_docs_hp_with_negatives_df = add_controlled_negative_documents(
    positive_span_docs_df=span_docs_hp_df,
    corpus_df=df,
    dataset_name="dish_ner_weak_high_precision_with_negatives",
    negative_ratio=0.10,
    max_negatives_per_split=1500
)

show_span_dataset_distribution(
    span_docs_hp_with_negatives_df,
    "HIGH PRECISION + CONTROLLED NEGATIVES"
)

HIGH PRECISION + CONTROLLED NEGATIVES
Documentos: 43161
Entidades: 98158

Split counts:


split
train         34182
validation     4502
test           4477
Name: count, dtype: int64


Sentiment counts:


sentiment_label
positive    30246
negative     7193
neutral      5722
Name: count, dtype: int64


Split x sentiment:


sentiment_label,negative,neutral,positive
split,,,
test,714,567,3196
train,5714,4528,23940
validation,765,627,3110



Entity count per doc:


count    43161.000000
mean         2.274229
std          2.041167
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         37.000000
Name: entity_count, dtype: float64

In [17]:
# ============================================================
# 15. Muestra para revisión manual de spans
# ============================================================

def render_text_with_entities(text: str, entities: list[dict]) -> str:
    """
    Render simple para revisar entidades entre corchetes.
    """

    if not entities:
        return text

    entities_sorted = sorted(entities, key=lambda e: e["start"])

    output = []
    last = 0

    for ent in entities_sorted:
        start = ent["start"]
        end = ent["end"]

        output.append(text[last:start])
        output.append("[DISH:")
        output.append(text[start:end])
        output.append("]")
        last = end

    output.append(text[last:])

    return "".join(output)


review_sample_parts = []

for label in ["negative", "neutral", "positive"]:
    subset = span_docs_hp_df[span_docs_hp_df["sentiment_label"] == label].copy()

    subset = subset.sort_values("entity_count", ascending=False)

    review_sample_parts.append(subset.head(30))

manual_span_review_df = pd.concat(review_sample_parts, ignore_index=True)

manual_span_review_df["text_with_entities"] = manual_span_review_df.apply(
    lambda row: render_text_with_entities(row["text"], row["entities"]),
    axis=1
)

manual_span_review_df = manual_span_review_df[
    [
        "corpus_document_id",
        "review_id",
        "business_id",
        "split",
        "rating_value",
        "sentiment_label",
        "entity_count",
        "entities",
        "text",
        "text_with_entities",
        "dataset_name",
        "labeling_method",
        "human_review_status",
    ]
].copy()

print("Muestra manual:", len(manual_span_review_df))
display(manual_span_review_df.head(10))

Muestra manual: 90


,corpus_document_id,review_id,business_id,split,rating_value,sentiment_label,entity_count,entities,text,text_with_entities,dataset_name,labeling_method,human_review_status
0,yelp_2c8e4a3dac86656c98c5b160,fIme7Ys0imjvIqPVWdX-Rw,M0r9lUn2gLFYgIwIfG8-bQ,train,1.0,negative,28,"[{'start': 71, 'end': 77, 'label': 'DISH', 'text': 'burger', 'normalized_text': 'burger', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 427, 'end': 433, 'label': 'DISH', 'text': 'burger', 'normalized_text': 'burger', 'candidat...","Sigh...This place has to be the most hyped-up overrated, ""slider"" (not burger) place I have ever eaten at. No skill, no flavor, and no creativity coming out this kitchen for sure. Bailey's, get it together please! Pickles were pretty good, however, my preference for fried foods is breadcrumbs, n...","Sigh...This place has to be the most hyped-up overrated, ""slider"" (not [DISH:burger]) place I have ever eaten at. No skill, no flavor, and no creativity coming out this kitchen for sure. Bailey's, get it together please! Pickles were pretty good, however, my preference for fried foods is breadcr...",dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
1,yelp_62a895d578da20b0afc6fdb1,a2n8Jp5rnofTmu3KKppMIQ,8-AIbFaP6GP9BwHZdclBog,train,2.0,negative,21,"[{'start': 67, 'end': 72, 'label': 'DISH', 'text': 'sushi', 'normalized_text': 'sushi', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 84, 'end': 89, 'label': 'DISH', 'text': 'sushi', 'normalized_text': 'sushi', 'candidate_gran...","Remember my review on Sengyo?? I just rather go there or try other sushi places for sushi instead of Oh Yoko. But before I get into that: First off, I've ordered Pork Katsu as take-out twice here before - once for lunch (I think it was $11) and once crazily for dinner ($20-something?). Don't get...","Remember my review on Sengyo?? I just rather go there or try other [DISH:sushi] places for [DISH:sushi] instead of Oh Yoko. But before I get into that: First off, I've ordered Pork Katsu as take-out twice here before - once for lunch (I think it was $11) and once crazily for dinner ($20-somethin...",dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
2,yelp_cc331c11ea46284f33bda8d7,_li1ZMMXYWC2_H2R9ErpXA,VYTFqb1NVLleeStQaUEP0Q,validation,2.0,negative,20,"[{'start': 26, 'end': 32, 'label': 'DISH', 'text': 'Burger', 'normalized_text': 'burger', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 235, 'end': 241, 'label': 'DISH', 'text': 'Burger', 'normalized_text': 'burger', 'candidat...","I've actually been to the Burger Bar before, but this was during my pre-Yelp days (and pre-meat eating days; yes, I was a vegetarian for seven long years!) So I'm going to start a fresh review based on my most recent experience at the Burger Bar. Me and Andy visited the Burger Bar on a weekday n...","I've actually been to the [DISH:Burger] Bar before, but this was during my pre-Yelp days (and pre-meat eating days; yes, I was a vegetarian for seven long years!) So I'm going to start a fresh review based on my most recent experience at the [DISH:Burger] Bar. Me and Andy visited the [DISH:Burge...",dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
3,yelp_27f420da8d25193de755bc85,Mm2FOIieLru-BdZXMgvTrA,PW6C4OMpPQ_Zm7oiwNIVgw,train,1.0,negative,18,"[{'start': 512, 'end': 518, 'label': 'DISH', 'text': 'Salmon', 'normalized_text': 'salmon', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 878, 'end': 885, 'label': 'DISH', 'text': 'gnocchi', 'normalized_text': 'gnocchi', 'cand...","It was a Saturday night and market st in Philly was swarming with people. My friends and I checked 2 restaurants but they both had over an hour long waits. The wait at Anjou? Non existant. That should have been a

In [18]:
# ============================================================
# 16. Guardar datasets de spans
# ============================================================

all_v2_spans_path = SPANS_DIR / "dish_ner_weak_spans_all_v2.jsonl"
hp_spans_path = SPANS_DIR / "dish_ner_weak_spans_high_precision.jsonl"
hp_neg_spans_path = SPANS_DIR / "dish_ner_weak_spans_high_precision_with_negatives.jsonl"
manual_span_review_path = SAMPLES_DIR / "dish_ner_manual_span_review_sample_90.jsonl"

save_jsonl(span_docs_all_v2_df, all_v2_spans_path)
save_jsonl(span_docs_hp_df, hp_spans_path)
save_jsonl(span_docs_hp_with_negatives_df, hp_neg_spans_path)
save_jsonl(manual_span_review_df, manual_span_review_path)

summary = {
    "notebook": "05_dish_ner_dataset_builder",
    "stage": "span_dataset_builder",
    "source_files": {
        "corpus": str(CORPUS_PATH),
        "weak_candidates_v2": str(WEAK_CANDIDATES_V2_PATH),
        "dish_lexicon": str(DISH_LEXICON_PATH),
    },
    "settings": {
        "high_precision_granularities": sorted(list(HIGH_PRECISION_GRANULARITIES)),
        "high_precision_min_confidence": HIGH_PRECISION_MIN_CONFIDENCE,
        "controlled_negative_ratio": 0.10,
    },
    "datasets": {
        "all_v2": qa_all_v2,
        "high_precision": qa_hp,
        "high_precision_with_negatives": {
            "total_docs": int(len(span_docs_hp_with_negatives_df)),
            "total_entities": int(span_docs_hp_with_negatives_df["entity_count"].sum()),
            "docs_without_entities": int((span_docs_hp_with_negatives_df["entity_count"] == 0).sum()),
            "split_counts": {
                str(k): int(v)
                for k, v in span_docs_hp_with_negatives_df["split"].value_counts().to_dict().items()
            },
        }
    },
    "outputs": {
        "all_v2_spans": str(all_v2_spans_path),
        "high_precision_spans": str(hp_spans_path),
        "high_precision_with_negatives_spans": str(hp_neg_spans_path),
        "manual_span_review_sample": str(manual_span_review_path),
    }
}

summary_path = ARTIFACTS_DIR / "dish_ner_span_dataset_summary.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Resumen guardado en:")
print(summary_path)

print(json.dumps(summary, indent=2, ensure_ascii=False)[:4000])

Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/spans/dish_ner_weak_spans_all_v2.jsonl (49,189 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/spans/dish_ner_weak_spans_high_precision.jsonl (40,845 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/spans/dish_ner_weak_spans_high_precision_with_negatives.jsonl (43,161 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/samples/dish_ner_manual_span_review_sample_90.jsonl (90 registros)
Resumen guardado en:
/kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/artifacts/dish_ner_span_dataset_summary.json
{
  "notebook": "05_dish_ner_dataset_builder",
  "stage": "span_dataset_builder",
  "source_files": {
    "corpus": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_food_reviews_corpus_sample_100k_lines.jsonl",
    "weak_candidates_v2": "/kaggle/input/datasets/ivanarteagacordero/hi

## 2. Conversión de spans a formato BIO

Una vez construidos los documentos con spans `DISH`, el siguiente paso es convertirlos a formato token-level.

El formato BIO representa cada token con una etiqueta:

- `O`: token fuera de una entidad.
- `B-DISH`: primer token de una mención de plato.
- `I-DISH`: token interno de una mención de plato.

Ejemplo:

```text
The crab legs were amazing
O   B-DISH I-DISH O    O

```
Este formato será la base para entrenar un modelo NER / token classification con Hugging Face.

In [19]:
# ============================================================
# 17. Definir etiquetas BIO
# ============================================================

NER_LABELS = ["O", "B-DISH", "I-DISH"]

label2id = {
    label: idx
    for idx, label in enumerate(NER_LABELS)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print("Labels:")
print(label2id)
print(id2label)

Labels:
{'O': 0, 'B-DISH': 1, 'I-DISH': 2}
{0: 'O', 1: 'B-DISH', 2: 'I-DISH'}


In [20]:
# ============================================================
# 18. Tokenizador simple con offsets
# ============================================================

TOKEN_PATTERN = re.compile(
    r"\w+(?:[-']\w+)*|[^\w\s]",
    flags=re.UNICODE
)

def tokenize_with_offsets(text: str) -> list[dict]:
    """
    Tokeniza un texto manteniendo offsets de caracteres.
    Esta tokenización es suficiente para construir un dataset BIO intermedio.
    En el notebook de entrenamiento, estos tokens se alinearán con el tokenizer del transformer.
    """

    if not isinstance(text, str):
        return []

    tokens = []

    for match in TOKEN_PATTERN.finditer(text):
        tokens.append({
            "text": match.group(0),
            "start": int(match.start()),
            "end": int(match.end()),
        })

    return tokens


# Prueba rápida
sample_text = "The crab legs were amazing, but the fries were cold."
sample_tokens = tokenize_with_offsets(sample_text)

display(pd.DataFrame(sample_tokens))

,text,start,end
0,The,0,3
1,crab,4,8
2,legs,9,13
3,were,14,18
4,amazing,19,26
5,",",26,27
6,but,28,31
7,the,32,35
8,fries,36,41
9,were,42,46


In [21]:
# ============================================================
# 19. Funciones para asignar etiquetas BIO
# ============================================================

def token_overlaps_entity(token: dict, entity: dict) -> bool:
    """
    Comprueba si un token solapa con una entidad.
    """

    return not (
        token["end"] <= entity["start"]
        or entity["end"] <= token["start"]
    )


def assign_bio_labels(tokens: list[dict], entities: list[dict]) -> list[str]:
    """
    Asigna etiquetas BIO a una lista de tokens usando entidades con offsets.
    """

    labels = ["O"] * len(tokens)

    if not entities:
        return labels

    entities_sorted = sorted(
        entities,
        key=lambda e: (int(e["start"]), int(e["end"]))
    )

    for entity in entities_sorted:
        entity_token_indices = []

        for idx, token in enumerate(tokens):
            if token_overlaps_entity(token, entity):
                entity_token_indices.append(idx)

        if not entity_token_indices:
            continue

        first_idx = entity_token_indices[0]

        # No sobrescribir entidades ya asignadas
        if labels[first_idx] == "O":
            labels[first_idx] = "B-DISH"

        for idx in entity_token_indices[1:]:
            if labels[idx] == "O":
                labels[idx] = "I-DISH"

    return labels


def convert_document_to_bio(row) -> dict:
    """
    Convierte un documento con spans a un registro BIO.
    """

    text = row["text"]
    entities = row["entities"]

    if not isinstance(entities, list):
        entities = []

    tokens_with_offsets = tokenize_with_offsets(text)
    bio_labels = assign_bio_labels(tokens_with_offsets, entities)

    tokens = [token["text"] for token in tokens_with_offsets]
    token_offsets = [
        {
            "start": int(token["start"]),
            "end": int(token["end"]),
        }
        for token in tokens_with_offsets
    ]

    ner_tags = [label2id[label] for label in bio_labels]

    return {
        "corpus_document_id": row["corpus_document_id"],
        "review_id": row["review_id"],
        "business_id": row["business_id"],
        "split": row["split"],
        "rating_value": make_json_safe(row["rating_value"]),
        "sentiment_label": row["sentiment_label"],
        "text": text,
        "tokens": tokens,
        "token_offsets": token_offsets,
        "ner_tags": ner_tags,
        "ner_tag_names": bio_labels,
        "entities": entities,
        "entity_count": int(row["entity_count"]),
        "token_count": len(tokens),
        "dish_token_count": int(sum(1 for label in bio_labels if label != "O")),
        "dataset_name": row["dataset_name"],
        "labeling_method": row["labeling_method"],
        "human_review_status": row["human_review_status"],
    }

In [22]:
# ============================================================
# 20. Prueba de conversión BIO
# ============================================================

sample_row = span_docs_hp_with_negatives_df[
    span_docs_hp_with_negatives_df["entity_count"] > 0
].iloc[0]

sample_bio = convert_document_to_bio(sample_row)

print("Texto:")
print(sample_bio["text"][:1000])

print("\nTokens + etiquetas:")
display(
    pd.DataFrame({
        "token": sample_bio["tokens"],
        "offset": sample_bio["token_offsets"],
        "label": sample_bio["ner_tag_names"],
    }).head(80)
)

print("\nEntidades:")
print(json.dumps(sample_bio["entities"], indent=2, ensure_ascii=False))

Texto:
Since going paleo it's hard to find something that doesn't taste like cardboard to satisfy my pizza cravings. The gluten free pizza here is the best I've found and they even personalize it any way I want including no cheese and extra sauce. It's my new fav!

Tokens + etiquetas:


,token,offset,label
0,Since,"{'start': 0, 'end': 5}",O
1,going,"{'start': 6, 'end': 11}",O
2,paleo,"{'start': 12, 'end': 17}",O
3,it's,"{'start': 18, 'end': 22}",O
4,hard,"{'start': 23, 'end': 27}",O
5,to,"{'start': 28, 'end': 30}",O
6,find,"{'start': 31, 'end': 35}",O
7,something,"{'start': 36, 'end': 45}",O
8,that,"{'start': 46, 'end': 50}",O
9,doesn't,"{'start': 51, 'end': 58}",O



Entidades:
[
  {
    "start": 94,
    "end": 99,
    "label": "DISH",
    "text": "pizza",
    "normalized_text": "pizza",
    "candidate_granularity": "dish_family",
    "candidate_sources": [
      "dictionary"
    ],
    "weak_confidence_v2": 0.75
  },
  {
    "start": 126,
    "end": 131,
    "label": "DISH",
    "text": "pizza",
    "normalized_text": "pizza",
    "candidate_granularity": "dish_family",
    "candidate_sources": [
      "dictionary"
    ],
    "weak_confidence_v2": 0.75
  }
]


In [23]:
# ============================================================
# 21. Convertir dataset completo high_precision_with_negatives a BIO
# ============================================================

bio_records = []

for row in tqdm(
    span_docs_hp_with_negatives_df.to_dict(orient="records"),
    desc="Convirtiendo documentos a BIO"
):
    bio_records.append(convert_document_to_bio(row))

bio_df = pd.DataFrame(bio_records)

print("BIO dataset shape:", bio_df.shape)

display(bio_df.head(3))

print("\nSplit counts:")
display(bio_df["split"].value_counts())

print("\nEntity count:")
display(bio_df["entity_count"].describe())

print("\nToken count:")
display(bio_df["token_count"].describe())

print("\nDish token count:")
display(bio_df["dish_token_count"].describe())

Convirtiendo documentos a BIO:   0%|          | 0/43161 [00:00<?, ?it/s]

BIO dataset shape: (43161, 18)


,corpus_document_id,review_id,business_id,split,rating_value,sentiment_label,text,tokens,token_offsets,ner_tags,ner_tag_names,entities,entity_count,token_count,dish_token_count,dataset_name,labeling_method,human_review_status
0,yelp_ce585eabced3468556e19c1a,RAhYHS028TKwAai7eFM4rA,ZWRgvtyHRobwpi4G-Z4STQ,test,5.0,positive,Since going paleo it's hard to find something that doesn't taste like cardboard to satisfy my pizza cravings. The gluten free pizza here is the best I've found and they even personalize it any way I want including no cheese and extra sauce. It's my new fav!,"[Since, going, paleo, it's, hard, to, find, something, that, doesn't, taste, like, cardboard, to, satisfy, my, pizza, cravings, ., The, gluten, free, pizza, here, is, the, best, I've, found, and, they, even, personalize, it, any, way, I, want, including, no, cheese, and, extra, sauce, ., It's, m...","[{'start': 0, 'end': 5}, {'start': 6, 'end': 11}, {'start': 12, 'end': 17}, {'start': 18, 'end': 22}, {'start': 23, 'end': 27}, {'start': 28, 'end': 30}, {'start': 31, 'end': 35}, {'start': 36, 'end': 45}, {'start': 46, 'end': 50}, {'start': 51, 'end': 58}, {'start': 59, 'end': 64}, {'start': 65...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[{'start': 94, 'end': 99, 'label': 'DISH', 'text': 'pizza', 'normalized_text': 'pizza', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 126, 'end': 131, 'label': 'DISH', 'text': 'pizza', 'normalized_text': 'pizza', 'candidate_gr...",2,50,2,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
1,yelp_3e0c6444441a9272c32f30ce,pKcQBSuGskizQ992yCRJTg,5iuo1kvv0XZMS0bUOoLz2Q,test,2.0,negative,"Wasn't too impressed with their food. Ordered the tuna tartare, which never came out :(. Had the sirloin steak, which wasn't even that good, and it was also overcooked. Fortunately I was with good company, so the food didn't bother me as much. However, I probably won't come back and have the foo...","[Wasn't, too, impressed, with, their, food, ., Ordered, the, tuna, tartare, ,, which, never, came, out, :, (, ., Had, the, sirloin, steak, ,, which, wasn't, even, that, good, ,, and, it, was, also, overcooked, ., Fortunately, I, was, with, good, company, ,, so, the, food, didn't, bother, me, as,...","[{'start': 0, 'end': 6}, {'start': 7, 'end': 10}, {'start': 11, 'end': 20}, {'start': 21, 'end': 25}, {'start': 26, 'end': 31}, {'start': 32, 'end': 36}, {'start': 36, 'end': 37}, {'start': 38, 'end': 45}, {'start': 46, 'end': 49}, {'start': 50, 'end': 54}, {'start': 55, 'end': 62}, {'start': 62...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[{'start': 50, 'end': 54, 'label': 'DISH', 'text': 'tuna', 'normalized_text': 'tuna', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 105, 'end': 110, 'label': 'DISH', 'text': 'steak', 'normalized_text': 'steak', 'candidate_gran...",2,66,2,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
2,yelp_ab3d21e21cc2584a47ab6510,aOb_YG545JLGJWPiEEY1xw,2Q1R2OhBbAQ581vK_r7NhA,train,5.0,positive,"Came here on a Monday night with a few friends. I don't know the area well, but I'm glad we decided on this place. We were seated quickly, and not even a little grumbling that they were closing in 45 minutes. The sushi was fresh. The food was in 


Split counts:


split
train         34182
validation     4502
test           4477
Name: count, dtype: int64

...

count    43161.000000
mean         2.274229
std          2.041167
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         37.000000
Name: entity_count, dtype: float64


Token count:


count    43161.000000
mean       132.719423
std        115.579425
min         12.000000
25%         56.000000
50%         97.000000
75%        170.000000
max       1170.000000
Name: token_count, dtype: float64


Dish token count:


count    43161.000000
mean         2.915479
std          2.679046
min          0.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         37.000000
Name: dish_token_count, dtype: float64

In [24]:
# ============================================================
# 22. QA del dataset BIO
# ============================================================

def validate_bio_dataset(bio_df: pd.DataFrame) -> dict:
    total_docs = len(bio_df)
    invalid_length_docs = 0
    invalid_tag_docs = 0
    i_without_previous_entity = 0
    docs_with_dish = 0
    docs_without_dish = 0

    label_counter = Counter()
    token_counter = 0

    for row in bio_df.itertuples(index=False):
        tokens = row.tokens
        ner_tags = row.ner_tags
        ner_tag_names = row.ner_tag_names

        if len(tokens) != len(ner_tags) or len(tokens) != len(ner_tag_names):
            invalid_length_docs += 1
            continue

        token_counter += len(tokens)

        has_dish = any(label != "O" for label in ner_tag_names)

        if has_dish:
            docs_with_dish += 1
        else:
            docs_without_dish += 1

        previous_label = "O"

        for label in ner_tag_names:
            label_counter[label] += 1

            if label not in NER_LABELS:
                invalid_tag_docs += 1

            if label == "I-DISH" and previous_label == "O":
                i_without_previous_entity += 1

            previous_label = label

    summary = {
        "total_docs": int(total_docs),
        "total_tokens": int(token_counter),
        "docs_with_dish": int(docs_with_dish),
        "docs_without_dish": int(docs_without_dish),
        "invalid_length_docs": int(invalid_length_docs),
        "invalid_tag_docs": int(invalid_tag_docs),
        "i_without_previous_entity_count": int(i_without_previous_entity),
        "label_counts": {
            str(k): int(v)
            for k, v in label_counter.items()
        },
        "label_percentages": {
            str(k): float(v / token_counter * 100) if token_counter else 0.0
            for k, v in label_counter.items()
        },
        "split_counts": {
            str(k): int(v)
            for k, v in bio_df["split"].value_counts().to_dict().items()
        },
        "token_count_summary": {
            "min": int(bio_df["token_count"].min()),
            "max": int(bio_df["token_count"].max()),
            "mean": float(bio_df["token_count"].mean()),
            "median": float(bio_df["token_count"].median()),
        },
        "entity_count_summary": {
            "min": int(bio_df["entity_count"].min()),
            "max": int(bio_df["entity_count"].max()),
            "mean": float(bio_df["entity_count"].mean()),
            "median": float(bio_df["entity_count"].median()),
        },
    }

    return summary


bio_qa_summary = validate_bio_dataset(bio_df)

print(json.dumps(bio_qa_summary, indent=2, ensure_ascii=False))

{
  "total_docs": 43161,
  "total_tokens": 5728303,
  "docs_with_dish": 40845,
  "docs_without_dish": 2316,
  "invalid_length_docs": 0,
  "invalid_tag_docs": 0,
  "i_without_previous_entity_count": 0,
  "label_counts": {
    "O": 5602468,
    "B-DISH": 98150,
    "I-DISH": 27685
  },
  "label_percentages": {
    "O": 97.80327611859917,
    "B-DISH": 1.7134219331624043,
    "I-DISH": 0.4833019482384225
  },
  "split_counts": {
    "train": 34182,
    "validation": 4502,
    "test": 4477
  },
  "token_count_summary": {
    "min": 12,
    "max": 1170,
    "mean": 132.71942262690854,
    "median": 97.0
  },
  "entity_count_summary": {
    "min": 0,
    "max": 37,
    "mean": 2.2742290493732766,
    "median": 2.0
  }
}


In [25]:
# ============================================================
# 23. Separar train / validation / test
# ============================================================

bio_train_df = bio_df[bio_df["split"] == "train"].copy()
bio_val_df = bio_df[bio_df["split"] == "validation"].copy()
bio_test_df = bio_df[bio_df["split"] == "test"].copy()

print("Train:", bio_train_df.shape)
print("Validation:", bio_val_df.shape)
print("Test:", bio_test_df.shape)

print("\nDocs con entidades por split:")
display(
    bio_df.assign(has_dish=bio_df["entity_count"] > 0)
    .groupby(["split", "has_dish"])
    .size()
    .reset_index(name="count")
)

Train: (34182, 18)
Validation: (4502, 18)
Test: (4477, 18)

Docs con entidades por split:


,split,has_dish,count
0,test,False,407
1,test,True,4070
2,train,False,1500
3,train,True,32682
4,validation,False,409
5,validation,True,4093


In [26]:
# ============================================================
# 24. Crear muestra pequeña para pruebas rápidas
# ============================================================

def create_smoke_test_bio_dataset(bio_df: pd.DataFrame, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """
    Crea una muestra pequeña para probar el entrenamiento NER rápidamente.
    Mantiene ejemplos con y sin entidades.
    """

    parts = []

    for split_name, n_with_dish, n_without_dish in [
        ("train", 800, 100),
        ("validation", 120, 30),
        ("test", 120, 30),
    ]:
        split_df = bio_df[bio_df["split"] == split_name].copy()

        with_dish = split_df[split_df["entity_count"] > 0]
        without_dish = split_df[split_df["entity_count"] == 0]

        if len(with_dish) > 0:
            parts.append(
                with_dish.sample(
                    n=min(n_with_dish, len(with_dish)),
                    random_state=random_state
                )
            )

        if len(without_dish) > 0:
            parts.append(
                without_dish.sample(
                    n=min(n_without_dish, len(without_dish)),
                    random_state=random_state
                )
            )

    smoke_df = pd.concat(parts, ignore_index=True)
    smoke_df = smoke_df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    return smoke_df


bio_smoke_df = create_smoke_test_bio_dataset(bio_df)

print("Smoke BIO dataset:", bio_smoke_df.shape)
display(bio_smoke_df["split"].value_counts())

display(
    bio_smoke_df.assign(has_dish=bio_smoke_df["entity_count"] > 0)
    .groupby(["split", "has_dish"])
    .size()
    .reset_index(name="count")
)

Smoke BIO dataset: (1200, 18)


split
train         900
test          150
validation    150
Name: count, dtype: int64

,split,has_dish,count
0,test,False,30
1,test,True,120
2,train,False,100
3,train,True,800
4,validation,False,30
5,validation,True,120


In [27]:
# ============================================================
# 25. Guardar datasets BIO
# ============================================================

bio_full_path = BIO_DIR / "dish_ner_bio_high_precision_with_negatives.jsonl"
bio_train_path = BIO_DIR / "dish_ner_bio_train.jsonl"
bio_val_path = BIO_DIR / "dish_ner_bio_validation.jsonl"
bio_test_path = BIO_DIR / "dish_ner_bio_test.jsonl"
bio_smoke_path = BIO_DIR / "dish_ner_bio_smoke_test.jsonl"

labels_path = ARTIFACTS_DIR / "dish_ner_labels.json"
bio_summary_path = ARTIFACTS_DIR / "dish_ner_bio_dataset_summary.json"

save_jsonl(bio_df, bio_full_path)
save_jsonl(bio_train_df, bio_train_path)
save_jsonl(bio_val_df, bio_val_path)
save_jsonl(bio_test_df, bio_test_path)
save_jsonl(bio_smoke_df, bio_smoke_path)

labels_config = {
    "labels": NER_LABELS,
    "label2id": label2id,
    "id2label": {
        str(k): v
        for k, v in id2label.items()
    }
}

with labels_path.open("w", encoding="utf-8") as f:
    json.dump(labels_config, f, indent=2, ensure_ascii=False)

bio_dataset_summary = {
    "notebook": "05_dish_ner_dataset_builder",
    "stage": "bio_dataset_builder",
    "source_span_dataset": str(hp_neg_spans_path),
    "bio_qa_summary": bio_qa_summary,
    "outputs": {
        "bio_full": str(bio_full_path),
        "bio_train": str(bio_train_path),
        "bio_validation": str(bio_val_path),
        "bio_test": str(bio_test_path),
        "bio_smoke_test": str(bio_smoke_path),
        "labels": str(labels_path),
    }
}

with bio_summary_path.open("w", encoding="utf-8") as f:
    json.dump(bio_dataset_summary, f, indent=2, ensure_ascii=False)

print("Datasets BIO guardados.")
print("Resumen BIO guardado en:")
print(bio_summary_path)

print(json.dumps(bio_dataset_summary, indent=2, ensure_ascii=False)[:4000])

Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/bio/dish_ner_bio_high_precision_with_negatives.jsonl (43,161 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/bio/dish_ner_bio_train.jsonl (34,182 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/bio/dish_ner_bio_validation.jsonl (4,502 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/bio/dish_ner_bio_test.jsonl (4,477 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/bio/dish_ner_bio_smoke_test.jsonl (1,200 registros)
Datasets BIO guardados.
Resumen BIO guardado en:
/kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/artifacts/dish_ner_bio_dataset_summary.json
{
  "notebook": "05_dish_ner_dataset_builder",
  "stage": "bio_dataset_builder",
  "source_span_dataset": "/kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/spans/dish_ner_weak_spans_high_p

In [28]:
# ============================================================
# 26. Render de ejemplos BIO para revisión
# ============================================================

def render_bio_tokens(tokens: list[str], labels: list[str]) -> str:
    """
    Renderiza tokens con entidades BIO en formato legible.
    """

    output = []
    inside_entity = False

    for token, label in zip(tokens, labels):
        if label == "B-DISH":
            if inside_entity:
                output.append("]")
            output.append("[DISH:")
            output.append(token)
            inside_entity = True

        elif label == "I-DISH":
            if inside_entity:
                output.append(" " + token)
            else:
                output.append("[DISH:" + token)
                inside_entity = True

        else:
            if inside_entity:
                output.append("]")
                inside_entity = False
            output.append(" " + token)

    if inside_entity:
        output.append("]")

    return "".join(output).strip()


review_examples = []

for label in ["negative", "neutral", "positive"]:
    subset = bio_df[
        (bio_df["sentiment_label"] == label)
        & (bio_df["entity_count"] > 0)
    ].copy()

    sample = subset.sample(
        n=min(5, len(subset)),
        random_state=RANDOM_STATE
    )

    for row in sample.itertuples(index=False):
        review_examples.append({
            "review_id": row.review_id,
            "split": row.split,
            "sentiment_label": row.sentiment_label,
            "entity_count": row.entity_count,
            "rendered_bio": render_bio_tokens(row.tokens, row.ner_tag_names),
        })

bio_review_examples_df = pd.DataFrame(review_examples)

display(bio_review_examples_df)

bio_review_examples_path = SAMPLES_DIR / "dish_ner_bio_rendered_examples.jsonl"
save_jsonl(bio_review_examples_df, bio_review_examples_path)

,review_id,split,sentiment_label,entity_count,rendered_bio
0,1r0TZiIjYonzHlUnjwVVYg,train,negative,1,"Went last night since we live nearby . Salsa was really good with lots of cilantro , just as I like it . Chips , however , were really greasy . We ordered the[DISH:chicken fajitas for 2] . Really disappointed by it . The chicken and rice lacked flavor . The guacamole should have been called onio..."
1,JeA_hOSDxe903mZ_0H-JAw,train,negative,3,"What a disappointment . We went in on a Friday evening a few weeks ago with a large group and were not at all impressed . I'll get to the positives first : 1 . It is one of the larger dining rooms on Passyunk Ave , so it was a good choice for a large group 2 . The bill was fairly inexpensive and..."
2,5wPq_FqKNx4NSdjW4MDCRQ,validation,negative,1,Hate to be harsh but dang ! After having a limited selection at terminal F I finally decided to go with Tony Luke's . I was looking forward in trying there Philly Cheese[DISH:Steak] been in Philly and all . So I unraveled the sandwich and to no surprise the sandwich I got didn't look nothing lik...
3,CPo1gtC_ZXSxRXeu6HjF1g,train,negative,3,"Eh , crust was dry and we got the Delmar which instead of[DISH:pizza] sauce they use bbq sauce however they barely put any on the[DISH:pizza] which caused it to be dry . The server was good . The[DISH:pizza] took forrrevvverrr to come out though we had been done with our salads for 20-30 minutes ."
4,o_zgeUfQUCvY_H-TYHKECQ,train,negative,11,"Had the swine heaven platter for 4 ppl last night . It was a hot evening and we weren't sure how hungry we were . The platter came with a full rack of[DISH:ribs] , 10 ounces of[DISH:brisket] , corn bread , coleslaw and beans . The only thing I had to comment on was that the beef[DISH:brisket] wa..."
5,fi6k71MlTgDlAb4f9gup_w,validation,neutral,4,"It must be good because there's always a line . . . . I would say if you can walk right in and eat , it's worth the wait . Otherwise , I wouldn't stand in line . I'm an regular[DISH:pancake] sort of guy . No fancy[DISH:pancakes] . Just[DISH:pancakes] , eggs and ham . Mine breakfast was good but ..."
6,_3ayekQ1ANuDhC9ffBeCYA,train,neutral,1,"Very good coffee . Love the Chai[DISH:latte] here . Excellent soups . Most of the pastries I've tried were great . Customer service is hit and miss ( more hits than misses ) . If you're short on time , go somewhere else ."
7,Of-FnItUDCe9sYR54_-87w,train,neutral,1,"I ordered the combo vermicelli and was pretty disappointed . The meat portion was very small compared to the huge portion of rice noodle they gave me make it look like there's a lot of food . The combo cake with 2 bland[DISH:shrimp] , 1 eggroll and a few pieces of marinated beef . I also ordered..."
8,q3gNr-00VeM2HJYrRULG3g,train,neutral,1,"The K & S World Market on Charlotte , like other markets of its kind , should definitely appeal to you if you are looking for good and low priced shopping , but might not be the place where you will find everything you need on your list . K & S is , nonetheless , large and a great place to find ..."
9,WxPB6parI4_9aR2Iqsx5Cg,train,neutral,1,"If you are looking for country cooking and not worried about calories , then this is your place . I go here maybe once a year and order anything I want with no care for calories . We recently went after a late night out . Grits , biscuits and gravy , along with some country fried[DISH:steak] was..."


Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_dataset_builder/samples/dish_ner_bio_rendered_examples.jsonl (15 registros)


## Cierre del Notebook 05

En este notebook se ha construido el primer dataset NER débil para detección de platos.

El flujo realizado ha sido:

1. Carga del corpus Yelp.
2. Carga de candidatos refinados v2.
3. Construcción de spans exactos `DISH`.
4. Validación de offsets y solapamientos.
5. Creación de dataset high precision.
6. Inclusión controlada de documentos sin entidades.
7. Conversión de spans a formato BIO.
8. Separación en train, validation y test.
9. Generación de una muestra pequeña para pruebas rápidas.

El dataset recomendado para el primer entrenamiento NER es:

`dish_ner_bio_high_precision_with_negatives.jsonl`

También se ha generado una muestra rápida:

`dish_ner_bio_smoke_test.jsonl`

que será útil para comprobar que el notebook de entrenamiento funciona antes de lanzar el entrenamiento completo.